In [ ]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/faster_rcnn
%cp VOC2007.zip /content
%cp VOC2012.zip /content
%cd /content

drive.flush_and_unmount()

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from src.backbone import backbone_transform

transform = backbone_transform

In [ ]:
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

train_dataloader = create_voc_dataloader(img_paths_list=voc2007_img_paths_train, transform=transform, batch_size=32, shuffle=True)
dataset = train_dataloader.dataset

In [ ]:
batch_imgs, batch_gt_boxes, batch_gt_labels, batch_img_sizes_before_pad = next(iter(train_dataloader))
batch_imgs.shape, batch_gt_boxes[0], batch_gt_labels[0], batch_img_sizes_before_pad[0]

In [ ]:
batch_gt_boxes

In [ ]:
from src.utils import display_random_images_with_annotations, plot_image_with_annotations

display_random_images_with_annotations(dataset)

In [ ]:
from src.backbone import Backbone
import torch

backbone = Backbone()
backbone.eval()  # Set the backbone to evaluation mode

backbone.to(device)  # Move the backbone to the appropriate device
batch_imgs = batch_imgs.to(device)
with torch.inference_mode():
    output = backbone(batch_imgs)
    print(f"Backbone output shape: {output.shape}")
    

In [ ]:
import torch
scales = torch.tensor([128, 256, 512])
ratios = torch.tensor([0.5, 1, 2])

widths = scales.view(-1, 1) * torch.sqrt(ratios).view(1, -1)
heights = scales.view(-1, 1) / torch.sqrt(ratios).view(1, -1)

widths.shape, heights.shape, torch.stack((widths.flatten(), heights.flatten()), dim=-1)[None, None, : ].shape  # This will give you the width and height pairs for each scale and ratio combination

In [ ]:
import torch
import torch.nn as nn

class RPN_Head(nn.Module):
    def __init__(self, in_channels, mid_channels):
        super(RPN_Head, self).__init__()
        self.num_anchors = 9
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1)
        self.conv_cls = nn.Conv2d(mid_channels, self.num_anchors * 2, kernel_size=1, stride=1)
        self.conv_reg = nn.Conv2d(mid_channels, self.num_anchors * 4, kernel_size=1, stride=1)


        self.conv1.weight.data.normal_(0, 0.01)
        self.conv_cls.weight.data.normal_(0, 0.01)
        self.conv_reg.weight.data.normal_(0, 0.01)
        self.conv1.bias.data.zero_()
        self.conv_cls.bias.data.zero_()
        self.conv_reg.bias.data.zero_()

        self.scales = torch.tensor([128, 256, 512])
        self.ratios = torch.tensor([0.5, 1, 2])

    def forward(self, x, batch_img_height=None, batch_img_width=None):
        self.img_height = batch_img_height
        self.img_width = batch_img_width
        
        x = torch.relu(self.conv1(x))
        batch_cls_logits = self.conv_cls(x).permute(0, 2, 3, 1)
        batch_box_deltas = self.conv_reg(x).permute(0, 2, 3, 1)

        batch_cls_logits = batch_cls_logits.reshape(batch_cls_logits.shape[0], -1, 2)
        batch_box_deltas = batch_box_deltas.reshape(batch_box_deltas.shape[0], -1, 4)

        batch_anchors = self.generate_anchors(x)

        return batch_cls_logits, batch_box_deltas, batch_anchors

    def generate_anchors(self, feature_map):
        # This function should generate anchors based on the feature map size and predefined scales/aspect ratios

        batch_size, _, width, height = feature_map.shape

        # Calculating stride for the feature map relative to the input image
        total_stride_x = self.img_height // height   
        total_stride_y = self.img_width // width    

        # For each position in the feature map, calculate the center of the anchor box
        # center_x = (x_coord * total_stride_x) + (0.5 * total_stride_x)
        # center_y = (y_coord * total_stride_y) + (0.5 * total_stride_y)

        # Create a grid of center positions
        grid_x = torch.arange(width).float() + 0.5
        grid_y = torch.arange(height).float() + 0.5

        # Create a meshgrid of x_center, y_center positions for each anchor for each value of width and height in feature map
        grid_x, grid_y = torch.meshgrid(grid_x, grid_y, indexing='ij')

        # Converting grid positions to the scale from the feature map to the image size
        grid_x = grid_x * total_stride_x
        grid_y = grid_y * total_stride_y

        # Now, for each center position, we need to generate anchors based on the scales and aspect ratios
        widths = self.scales.view(-1, 1) * torch.sqrt(self.ratios).view(1, -1)
        heights = self.scales.view(-1, 1) / torch.sqrt(self.ratios).view(1, -1)


        centres = torch.stack((grid_x, grid_y), dim=-1).unsqueeze(dim=-2)  # Shape: [height, width, 1, 2]
        sizes = torch.stack((heights.flatten(), widths.flatten()), dim=-1)[None, None, : ]  # Shape: [1, 1, num_scales * num_ratios, 2]

        anchors = torch.cat((centres.expand(-1, -1, sizes.shape[2], -1), sizes.expand(centres.shape[0], centres.shape[1], -1, -1)), dim=-1)  # Shape: [height, width, num_anchors, 4]
        anchors = anchors.view(-1, 4)  # Flatten to [num_anchors_total, 4]
        
        batch_anchors = anchors.repeat(batch_size, 1, 1)  # Repeat for each image in the batch
        batch_anchors = batch_anchors.to(feature_map.device)  # Move to the same device as the feature map
        
        return batch_anchors 
           

In [ ]:
test = RPN_Head(in_channels=1024, mid_channels=512)
print(batch_imgs.shape)

with torch.inference_mode():
    test.eval()
    test.to(device)

    forward = test.forward(output, batch_img_height=batch_imgs[0].shape[1], batch_img_width=batch_imgs[0].shape[2])  # Simulated original image sizes for the batch
    print(f"RPN cls logits shape: {forward[0].shape}, RPN reg preds shape: {forward[1].shape}")
    print(f"RPN anchors shape: {forward[2].shape}")

In [ ]:
def compute_iou_matrix(anchors, gt_box):
        # Compute the IoU matrix between anchors and ground truth boxes
        # anchors: [num_anchors, 4], gt_boxes: [num_gt_boxes, 4]
        # Returns: IoU matrix of shape [num_anchors, num_gt_boxes]


        # Convert anchors and gt_boxes to (x1, y1, x2, y2) format
        anchor_x1 = anchors[:, 0] - anchors[:, 2] / 2
        anchor_y1 = anchors[:, 1] - anchors[:, 3] / 2
        anchor_x2 = anchors[:, 0] + anchors[:, 2] / 2
        anchor_y2 = anchors[:, 1] + anchors[:, 3] / 2
        print(f"Shapes {anchor_x1.shape}, {anchor_y1.shape}, {anchor_x2.shape}, {anchor_y2.shape}")
        
        gt_x1 = gt_boxes[:, 0]
        gt_y1 = gt_boxes[:, 1]
        gt_x2 = gt_boxes[:, 2]
        gt_y2 = gt_boxes[:, 3]
        print(f"Shapes {gt_x1.shape}, {gt_y1.shape}, {gt_x2.shape}, {gt_y2.shape}")
        print(anchors.device, gt_boxes.device, anchor_x1.device, gt_x1.device)
        
        # Calculate intersection
        inter_x1 = torch.max(anchor_x1.unsqueeze(1), gt_x1.unsqueeze(0))                # Unsqueeze to align dimensions for broadcasting
        inter_y1 = torch.max(anchor_y1.unsqueeze(1), gt_y1.unsqueeze(0))
        inter_x2 = torch.min(anchor_x2.unsqueeze(1), gt_x2.unsqueeze(0))
        inter_y2 = torch.min(anchor_y2.unsqueeze(1), gt_y2.unsqueeze(0))
        print(f"Shapes {inter_x1.shape}, {inter_y1.shape}, {inter_x2.shape}, {inter_y2.shape}")

        

        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        print(inter_area.shape)
        
        # Calculate union
        anchor_area = (anchor_x2 - anchor_x1) * (anchor_y2 - anchor_y1)
        gt_area = (gt_x2 - gt_x1) * (gt_y2 - gt_y1)

        print(f"Shapes {anchor_area.shape}, {gt_area.shape}")
        
        union_area = anchor_area.unsqueeze(dim=1) + gt_area.unsqueeze(dim=0) - inter_area       # unsqueeze to align dimensions for broadcasting
        
        # Compute IoU
        iou_matrix = inter_area / union_area
        
        return iou_matrix

anchors = forward[2][0]  # Assuming forward is the output from RPN_Head's forward method
gt_boxes = batch_gt_boxes[0]  # Ensure batch_gt_boxes is on the same device as anchors
gt_boxes = gt_boxes.to(anchors.device)  # Move gt_boxes to the same device as anchors
iou_matrix = compute_iou_matrix(anchors, gt_boxes)

In [ ]:
class RPN_Loss(nn.Module):
    def __init__(self):
        super(RPN_Loss, self).__init__()
        self.cls_loss_fn = nn.CrossEntropyLoss()
        self.reg_loss_fn = nn.SmoothL1Loss()

    def forward(self, batch_cls_logits, batch_box_deltas, batch_anchors, batch_gt_boxes, img_sizes_before_pad):
        batch_size = batch_cls_logits.shape[0]

        total_cls_loss = 0.0
        total_reg_loss = 0.0

        for i in range(batch_size):
            img_height, img_width = img_sizes_before_pad[i]

            cls_logits = batch_cls_logits[i]
            box_deltas = batch_box_deltas[i]
            anchors = batch_anchors[i]
            gt_boxes = batch_gt_boxes[i]

            inside_indices = self.anchors_inside_image(anchors, img_height, img_width)

            anchors = anchors[inside_indices]
            cls_logits = cls_logits[inside_indices]
            box_deltas = box_deltas[inside_indices]

            anchor_labels = self.anchor_labelling(anchors, gt_boxes)

            sample_mask, sampled_pos_idx, sampled_neg_idx = self.create_sample_mask_per_img(anchor_labels)

            cls_loss = self.cls_loss_fn_per_img(cls_logits, anchor_labels, sample_mask)
            reg_loss = self.reg_loss_fn_per_img(box_deltas, gt_boxes, anchors, sampled_pos_idx)  # Assuming gt_boxes are the regression targets

            total_cls_loss += cls_loss
            total_reg_loss += reg_loss


    def cls_loss_fn_per_img(self, cls_logits, anchor_labels, sample_mask):
        selected_cls_logits = cls_logits[sample_mask]
        selected_anchor_labels = anchor_labels[sample_mask]

        targets = (selected_anchor_labels == 1).long()  # Convert to binary targets (1 for positive, 0 for negative)

        cls_loss = nn.functional.cross_entropy(selected_cls_logits, targets, reduction='mean')      # reduction='mean' to get the average loss over the selected samples i.e., divide by num_samples = Ncls
        return cls_loss
    
    def corners_to_center(boxes):
        x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        w = x2 - x1
        h = y2 - y1
        x_c = x1 + w / 2
        y_c = y1 + h / 2
        return torch.stack([x_c, y_c, w, h], dim=1)


    def reg_loss_fn_per_img(self, box_deltas, gt_boxes, valid_anchors, sampled_pos_idx):
        pred_box_deltas = box_deltas[sampled_pos_idx]
        selected_gt_boxes = gt_boxes[sampled_pos_idx]
        anchors = valid_anchors[sampled_pos_idx]

        gt_boxes_centered = self.corners_to_center(selected_gt_boxes)

        xc_a, yc_a, h_a, w_a = anchors[:, 0], anchors[:, 1], anchors[:, 2], anchors[:, 3]
        gt_xc, gt_yc, gt_h, gt_w = gt_boxes_centered[:, 0], gt_boxes_centered[:, 1], gt_boxes_centered[:, 2], gt_boxes_centered[:, 3]

        # Calculate the regression targets (deltas) for the selected anchors
        target_dx = (gt_xc - xc_a) / w_a
        target_dy = (gt_yc - yc_a) / h_a
        target_dw = torch.log(gt_w / w_a)
        target_dh = torch.log(gt_h / h_a)

        target_box_deltas = torch.stack((target_dx, target_dy, target_dw, target_dh), dim=1)

        reg_loss = nn.functional.smooth_l1_loss(pred_box_deltas, target_box_deltas, reduction='mean')      # reduction='mean' to get the average loss over the selected samples i.e., divide by num_samples = Nreg
        return reg_loss

    def create_sample_mask_per_img(self, anchor_labels, num_samples=256, pos_fraction=0.5):
        positive_idx = torch.where(anchor_labels == 1)[0]       # ...[0] as it is a single valued tuple
        negative_idx = torch.where(anchor_labels == -1)[0]

        num_pos = min(int(num_samples * pos_fraction), positive_idx.numel())    # numel() returns the number of elements in the tensor
        num_neg = min(num_samples - num_pos, negative_idx.numel())

        perm_pos = torch.randperm(positive_idx.numel(), device=anchor_labels.device)[:num_pos]
        perm_neg = torch.randperm(negative_idx.numel(), device=anchor_labels.device)[:num_neg]

        sampled_pos_idx = positive_idx[perm_pos]
        sampled_neg_idx = negative_idx[perm_neg]

        # Ignoring i.e., making the anchor labels of the unsampled anchors to 0 (ignore)
        sample_mask = torch.zeros_like(anchor_labels, dtype=torch.bool, device=anchor_labels.device)
        sample_mask[sampled_pos_idx] = True
        sample_mask[sampled_neg_idx] = True

        return sample_mask, sampled_pos_idx, sampled_neg_idx

    def anchor_labelling(self, anchors, gt_boxes, pos_iou_threshold=0.7, neg_iou_threshold=0.3):
        iou_matrix = self.compute_iou_matrix(anchors, gt_boxes)

        num_anchors, num_gt_boxes = iou_matrix.shape
        labels = torch.zeros((num_anchors,), dtype=torch.long, device=anchors.device)  # Initialize all labels to 0 (ignore)

        max_iou_per_anchor, matched_gt_indices = iou_matrix.max(dim=1)

        labels[max_iou_per_anchor < neg_iou_threshold] = -1  # Negative labels
        labels[max_iou_per_anchor >= pos_iou_threshold] = 1  # Positive labels

        max_iou_per_gt, matched_anchor_indices = iou_matrix.max(dim=0)

        labels[matched_anchor_indices] = 1  # Ensure each gt box has at least one positive anchor

        return labels

    def anchors_inside_image(self, anchors, img_height, img_width):
        # Check if anchors are inside the image boundaries
        xc, yc, h, w = anchors[:, 0], anchors[:, 1], anchors[:, 2], anchors[:, 3]

        x1 = xc - w / 2
        y1 = yc - h / 2
        x2 = xc + w / 2
        y2 = yc + h / 2

        inside_indices = (0<=x1) & (0<=y1) & (x2 < img_width) & (y2 < img_height)           # anchors[:, 0] is xc, anchors[:, 1] is yc, anchors[:, 2] is h, anchors[:, 3] is w
        return inside_indices
    
    def compute_iou_matrix(self, anchors, gt_boxes):
        # Compute the IoU matrix between anchors and ground truth boxes
        # anchors: [num_anchors, 4], gt_boxes: [num_gt_boxes, 4]
        # Returns: IoU matrix of shape [num_anchors, num_gt_boxes]
        
        # Convert anchors and gt_boxes to (x1, y1, x2, y2) format
        anchor_x1 = anchors[:, 0] - anchors[:, 2] / 2
        anchor_y1 = anchors[:, 1] - anchors[:, 3] / 2
        anchor_x2 = anchors[:, 0] + anchors[:, 2] / 2
        anchor_y2 = anchors[:, 1] + anchors[:, 3] / 2
        
        gt_x1 = gt_boxes[:, 0]
        gt_y1 = gt_boxes[:, 1]
        gt_x2 = gt_boxes[:, 2]
        gt_y2 = gt_boxes[:, 3]
        
        # Calculate intersection
        inter_x1 = torch.max(anchor_x1.unsqueeze(1), gt_x1.unsqueeze(0))        # Unsqueeze to align dimensions for broadcasting
        inter_y1 = torch.max(anchor_y1.unsqueeze(1), gt_y1.unsqueeze(0))
        inter_x2 = torch.min(anchor_x2.unsqueeze(1), gt_x2.unsqueeze(0))
        inter_y2 = torch.min(anchor_y2.unsqueeze(1), gt_y2.unsqueeze(0))
        
        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        
        # Calculate union
        anchor_area = (anchor_x2 - anchor_x1) * (anchor_y2 - anchor_y1)
        gt_area = (gt_x2 - gt_x1) * (gt_y2 - gt_y1)
        
        union_area = anchor_area.unsqueeze(dim=1) + gt_area.unsqueeze(dim=0) - inter_area           # Unsqueeze to align dimensions for broadcasting
        
        # Compute IoU
        iou_matrix = inter_area / union_area
        
        return iou_matrix
